# FinDoc RAG: Banking & Regulatory Document Assistant
**Stack:** LangChain · FAISS · BM25 · sentence-transformers (BGE embeddings + cross-encoder reranker) · Groq (`openai/gpt-oss-120b`)

What this notebook does:
1. Loads finance documents (RBI circulars, bank annual reports, fee schedules / FAQs) with rich metadata
2. Builds a hybrid retriever (vector + BM25, fused with Reciprocal Rank Fusion) with optional cross-encoder reranking
3. Answers with a grounded prompt, page-level citations and an explicit refusal rule
4. **Evaluates** the system (retrieval hit rate, MRR, faithfulness, refusal rate) and compares retrieval strategies
5. Saves the index and serves a small Gradio UI

**Before you run:** add `GROQ_API_KEY` in Colab (key icon in the left sidebar → Secrets), or paste it when prompted.

## 1. Install

In [ ]:
!pip -q install -U langchain-core langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu pypdf sentence-transformers rank_bm25 pandas requests gradio tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69

## 2. Imports and config

In [ ]:
import os, re, json, time, pickle, random, getpass
from pathlib import Path

import requests
import pandas as pd
from IPython.display import display, Markdown

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

# ---- Config (tweak these and re-run the eval to compare) ----
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "openai/gpt-oss-120b"
FETCH_K = 30      # candidates pulled from each retriever
TOP_K = 5         # chunks sent to the LLM
REFUSAL = "The provided documents do not contain enough information to answer this question."

/tmp/ipykernel_548/99230818.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. API key and LLM

In [ ]:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

llm = ChatGroq(model=LLM_MODEL, temperature=0)

def safe_invoke(chain, inputs, retries=4):
    """Invoke with simple exponential backoff (handles free-tier rate limits)."""
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            wait = 2 ** attempt * 2
            print(f"LLM call failed ({type(e).__name__}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("LLM call failed after retries")

## 4. Get the documents
Use **either** option (or both). Good sources, all public:
- RBI Master Directions and circulars: https://rbi.org.in → *Notifications*
- Annual reports from a bank's investor-relations page (HDFC Bank, SBI, ICICI...)
- Fee schedules / FAQ PDFs from bank websites

Start with 5-10 PDFs. Annual reports are huge, so one or two is plenty.

In [ ]:
# Option A: upload PDFs from your computer
from google.colab import files

DOC_DIR = Path("finance_docs")
DOC_DIR.mkdir(exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    (DOC_DIR / name).write_bytes(data)
print("Saved:", list(uploaded.keys()))

Saving file7.pdf to file7.pdf
Saving file6.pdf to file6.pdf
Saving file5.pdf to file5.pdf
Saving file4.pdf to file4.pdf
Saving file3.pdf to file3.pdf
Saving file2.pdf to file2.pdf
Saving file1.pdf to file1.pdf
Saved: ['file7.pdf', 'file6.pdf', 'file5.pdf', 'file4.pdf', 'file3.pdf', 'file2.pdf', 'file1.pdf']


In [ ]:
# Option B: download from direct PDF links (fill in the dict; skip this cell if you used Option A)
PDF_URLS = {
    # "rbi_kyc_master_direction_2025.pdf": "https://rbidocs.rbi.org.in/.../your_file.pdf",
    # "hdfc_annual_report_2024.pdf": "https://.../annual-report.pdf",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (research project)"}
for fname, url in PDF_URLS.items():
    try:
        r = requests.get(url, headers=HEADERS, timeout=60)
        r.raise_for_status()
        if not r.content.startswith(b"%PDF"):
            print(f"Skipped {fname}: response is not a PDF (site may block bots, so download manually and use Option A)")
            continue
        (DOC_DIR / fname).write_bytes(r.content)
        print("Downloaded", fname)
    except Exception as e:
        print(f"Could not download {fname}: {e}")

print("PDFs available:", [p.name for p in DOC_DIR.glob('*.pdf')])

PDFs available: ['file4.pdf', 'file5.pdf', 'file6.pdf', 'file3.pdf', 'file2.pdf', 'file1.pdf', 'file7.pdf']


## 5. Load PDFs with finance-specific metadata
Metadata lets you filter retrieval, e.g. *"only circulars issued 2023 or later"*, which matters because regulations get superseded. Year and type are inferred heuristically from the filename and first page; override them in the dict below if they're wrong.

In [ ]:
# Manual overrides, e.g. {"rbi_kyc_master_direction_2025.pdf": {"doc_type": "circular", "year": 2025}}
METADATA_OVERRIDES = {}

def infer_doc_type(fname):
    f = fname.lower()
    if "annual" in f: return "annual_report"
    if "faq" in f: return "faq"
    if any(k in f for k in ["fee", "tariff", "charges"]): return "fee_schedule"
    if any(k in f for k in ["rbi", "direction", "master", "circular", "notification"]): return "circular"
    return "other"

def infer_year(first_page_text, fname):
    # (?<!\d) / (?!\d) instead of \b, so years next to underscores (rbi_cb_fraud_2026.pdf) are found
    pat = r"(?<!\d)(?:19|20)\d{2}(?!\d)"
    for text in (fname, first_page_text[:3000]):   # filename wins over page text
        years = [int(y) for y in re.findall(pat, text) if 1990 <= int(y) <= 2026]
        if years:
            return years[0]
    return None

docs = []
for pdf in sorted(DOC_DIR.glob("*.pdf")):
    pages = PyPDFLoader(str(pdf)).load()
    if not pages:
        continue
    meta = {
        "doc_name": pdf.name,
        "doc_type": infer_doc_type(pdf.name),
        "year": infer_year(pages[0].page_content, pdf.name),
    }
    meta.update(METADATA_OVERRIDES.get(pdf.name, {}))
    for p in pages:
        p.metadata.update(meta)
    docs.extend(pages)

assert docs, "No PDFs found. Run Option A or B above."
print(f"Loaded {len(docs)} pages from {len({d.metadata['doc_name'] for d in docs})} documents")
display(pd.DataFrame(
    list({(d.metadata["doc_name"], d.metadata["doc_type"], d.metadata["year"]) for d in docs}),
    columns=["doc_name", "doc_type", "year"]
).sort_values("doc_name"))

Loaded 227 pages from 7 documents


,doc_name,doc_type,year
5,file1.pdf,other,2026
4,file2.pdf,other,2026
0,file3.pdf,other,2025
3,file4.pdf,other,2026
1,file5.pdf,other,2026
2,file6.pdf,other,2026
6,file7.pdf,other,2022


## 6. Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i   # used later to measure retrieval accuracy
print(f"Created {len(chunks)} chunks")

Created 570 chunks


## 7. Indexes: FAISS (semantic) + BM25 (keyword) + cross-encoder reranker

In [ ]:
def tokenize(text):
    """Lowercase alphanumeric tokens, so 'Fraud,' 'fraud' and 'FRAUD' all match in BM25."""
    return re.findall(r"[a-z0-9]+", text.lower())

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)

bm25 = BM25Retriever.from_documents(chunks, k=FETCH_K, preprocess_func=tokenize)

reranker = CrossEncoder(RERANK_MODEL)
print("Indexes ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Indexes ready.


## 8. Retrieval
Four modes so you can compare them in the evaluation:
- `vector`: embeddings only
- `bm25`: keywords only (great for exact terms like *"Section 35A"* or *"KYC"*)
- `hybrid`: both lists merged with **Reciprocal Rank Fusion**
- `hybrid+rerank`: hybrid candidates re-scored by a cross-encoder

In [ ]:
def rrf(rank_lists, c=60):
    """Reciprocal Rank Fusion over several ranked lists of chunks."""
    scores, by_id = {}, {}
    for lst in rank_lists:
        for rank, d in enumerate(lst):
            cid = d.metadata["chunk_id"]
            scores[cid] = scores.get(cid, 0) + 1.0 / (c + rank + 1)
            by_id[cid] = d
    return [by_id[cid] for cid in sorted(scores, key=scores.get, reverse=True)]

def retrieve(query, mode="hybrid+rerank", k=TOP_K, doc_type=None, min_year=None):
    if mode == "vector":
        cands = vectorstore.similarity_search(query, k=FETCH_K)
    elif mode == "bm25":
        cands = bm25.invoke(query)
    elif mode in ("hybrid", "hybrid+rerank"):
        cands = rrf([vectorstore.similarity_search(query, k=FETCH_K), bm25.invoke(query)])
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # Metadata filters
    def keep(d):
        if doc_type and d.metadata.get("doc_type") != doc_type: return False
        if min_year and (d.metadata.get("year") or 0) < min_year: return False
        return True
    cands = [d for d in cands if keep(d)]

    if mode == "hybrid+rerank" and cands:
        cands = cands[:FETCH_K]
        scores = reranker.predict([(query, d.page_content) for d in cands])
        cands = [d for d, _ in sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)]

    return cands[:k]

## 9. Grounded prompt and chain (LCEL)

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
You are a banking and regulatory-compliance assistant.

Answer the question using ONLY the context below, which comes from RBI circulars,
bank annual reports and similar finance documents.

Rules:
- Do not invent facts, figures, dates, limits or circular numbers.
- If the context does not contain the answer, reply exactly: "{refusal}"
- Cite every claim with [Source N] matching the context blocks.
- When a document's year matters (regulations change), mention the year.
- Be concise and structured. Use bullet points for lists of requirements.

Question:
{question}

Context:
{context}
""")

# LCEL chain: prompt -> LLM -> plain string
rag_chain = rag_prompt | llm | StrOutputParser()

def format_docs(docs):
    blocks = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        blocks.append(
            f"[Source {i}] {m.get('doc_name')} | type: {m.get('doc_type')} | "
            f"year: {m.get('year')} | page: {m.get('page', 0) + 1}\n{d.page_content}"
        )
    return "\n\n---\n\n".join(blocks)

def answer_question(question, mode="hybrid+rerank", k=TOP_K, doc_type=None, min_year=None):
    docs = retrieve(question, mode=mode, k=k, doc_type=doc_type, min_year=min_year)
    if not docs:
        return {"answer": REFUSAL, "docs": []}
    answer = safe_invoke(rag_chain, {"question": question, "context": format_docs(docs), "refusal": REFUSAL})
    return {"answer": answer, "docs": docs}

def sources_table(docs):
    return pd.DataFrame([{
        "source": f"Source {i}",
        "document": d.metadata.get("doc_name"),
        "type": d.metadata.get("doc_type"),
        "year": d.metadata.get("year"),
        "page": d.metadata.get("page", 0) + 1,
    } for i, d in enumerate(docs, 1)])

def ask(question, **kwargs):
    out = answer_question(question, **kwargs)
    display(Markdown("### Answer"))
    display(Markdown(out["answer"]))
    if out["docs"]:
        display(Markdown("### Sources"))
        display(sources_table(out["docs"]))
    return out

## 10. Try it
Replace these with questions your documents can actually answer. The last two are deliberately out of scope: the bot should refuse.

In [ ]:
_ = ask("What are the responsibilities of the Board of a bank under the fraud risk management directions?")

### Answer

**Board responsibilities under the RBI Fraud Risk Management Directions (2026)**  

- **Approve a comprehensive Fraud Risk Management Policy** that:
  * Defines the roles and responsibilities of the Board (or Local Advisory Board), Board committees and Senior Management.  
  * Covers prevention, early detection, investigation, staff accountability, monitoring, recovery, reporting of frauds, and the framework for Early Warning Signals (EWS) and Red‑Flagging of Accounts (RFA).  
  * Incorporates measures to ensure compliance with the principles of natural justice.  [Source 1]

- **Set the scope and frequency of fraud‑review processes**:
  * Decide the coverage (e.g., fraud categories, trends, sectoral/geographic concentration, detection delays, staff‑accountability delays) and periodicity of reviews undertaken by the Senior Committee on Bank’s Management of Fraud (SCBMF). [Source 2]  
  * Determine the monetary threshold for fraud cases that must be escalated to the SCBMF, taking into account the bank’s scale and complexity. [Source 2]

- **Oversee the effectiveness of the fraud‑risk framework**:
  * Through the Risk Management Committee of the Board (RMCB), monitor the overall fraud‑risk management framework approved by the Board. [Source 3]

- **Govern the Early Warning Signals (EWS) and Red‑Flagging of Accounts (RFA) framework**:
  * Approve the EWS indicators to be monitored for credit facilities, loan accounts and other transactions. [Source 3]  
  * Prescribe the turnaround time for examining EWS alerts/triggers (preferably ≤ 30 days). [Source 3]  
  * Review the status of red‑flagged accounts and related EWS alerts. [Source 3]

- **Receive and act on periodic fraud‑incident reviews** presented by Senior Management to the Board or the Audit Committee of the Board (as appropriate). [Source 5]

These responsibilities collectively ensure that the Board provides strategic direction, oversight, and accountability for fraud risk management within the bank.

### Sources

,source,document,type,year,page
0,Source 1,file7.pdf,other,2022,6
1,Source 2,file7.pdf,other,2022,8
2,Source 3,file7.pdf,other,2022,9
3,Source 4,file7.pdf,other,2022,17
4,Source 5,file7.pdf,other,2022,8


In [21]:
_ = ask("What is the timeline for reporting a fraud to the Reserve Bank of India?")

### Answer

**Timeline for reporting a fraud to the RBI (as per the 2022 RBI “Fraud Risk Management” Directions)**  

- **Immediate reporting** – For frauds involving forged or fake instruments, the *presenting bank must hand over the instrument immediately* to the paying bank so that the latter can inform law‑enforcement agencies and report the fraud to the RBI. [Source 2]  

- **Incidents of theft, burglary, dacoity or robbery** – The bank must **report the case to the Fraud Monitoring Group, RBI, “immediately (not later than seven days) from their occurrence.”** [Source 3]  

- **Quarterly summary of such cases** – A **quarterly Return (RBR) must be submitted within 15 days after the end of the quarter** to which the cases relate. [Source 3]  

These are the only reporting time‑frames specified in the provided documents. If a different type of fraud is being considered, the same “immediate” approach (using the online Fraud Monitoring Returns portal) is implied, but a specific number of days is not given in the excerpts.

### Sources

,source,document,type,year,page
0,Source 1,file7.pdf,other,2022,18
1,Source 2,file7.pdf,other,2022,22
2,Source 3,file7.pdf,other,2022,24
3,Source 4,file7.pdf,other,2022,23
4,Source 5,file7.pdf,other,2022,1


In [22]:
_ = ask("What security controls must banks implement for digital payment transactions?")

### Answer

**Security controls required for digital‑payment transactions (RBI Directions 2026)**  

- **Server‑side integrity & authentication** – The bank’s server must enforce checks that prevent any transaction through non‑genuine or unauthorised payment products/applications and must use a robust, secure, centrally‑managed authentication process. [Source 1]

- **Application‑level data protection** – Controls must cover how payment applications **handle, store and protect payment data**; secure APIs for data storage and communication must be correctly implemented. The bank should follow OWASP‑MASVS, OWASP‑ASVS, ISO 12812 and other relevant OWASP standards and threat‑catalogues. [Source 1]

- **Web‑application firewall & DDoS protection** – Deploy a **Web Application Firewall (WAF)** and appropriate **DDoS‑mitigation techniques** for all internet‑facing digital‑payment products and services. [Source 2]

- **Cryptographic strength** – Use strong key lengths, algorithms, cipher suites, digital certificates and protocols for:
  * symmetric/asymmetric encryption,
  * hashing,
  * signing,
  * key exchange,
  * message‑digest creation,
  * random‑number generation,
  * authentication.  
  Adopt internationally accepted, non‑deprecated standards and keep configurations compliant with law. [Source 2]

- **Certificate management** – Renew all digital certificates used in the payment ecosystem well before expiry. [Source 2]

- **Multi‑factor authentication (MFA)** – Implement **robust MFA** to deter fraud, protect confidentiality of payment data and resist phishing, key‑logging, malware, etc.  
  * May use **adaptive authentication** that selects factors based on risk assessment, user profile and behaviour. [Source 3]

- **Customer alerts & OTPs** – OTP/alert messages for online transactions must display the **merchant name** (instead of only the aggregator) wherever applicable. [Source 4]

- **Mitigation of man‑in‑the‑middle (MITM) attacks** – As part of MFA, put in place measures to prevent MITM, man‑in‑the‑browser (MITB) and man‑in‑the‑application attacks; ensure data in transit is encrypted and that only genuine/authorised sources can authenticate transactions. [Source 4]

- **Session integrity** – An authenticated session, together with its encryption protocol, must remain intact for the whole customer interaction. [Source 4]

- **Mobile‑payment specific controls** – For mobile banking/payments:
  * Detect any anomaly or exception not programmed in the app; force the customer to **remove the current app instance and reinstall a verified version** before enabling transactions. [Source 5]
  * Enforce **device‑policy controls** that allow installation/execution of applications only after meeting defined baseline security requirements. [Source 5]

These controls collectively satisfy the RBI’s 2026 regulatory expectations for securing digital‑payment transactions.

### Sources

,source,document,type,year,page
0,Source 1,file6.pdf,other,2026,14
1,Source 2,file6.pdf,other,2026,11
2,Source 3,file6.pdf,other,2026,15
3,Source 4,file6.pdf,other,2026,16
4,Source 5,file6.pdf,other,2026,22


In [23]:
_ = ask("What governance requirements apply to a bank's technology risk and cybersecurity framework?")

### Answer

**Governance requirements for a bank’s technology‑risk and cyber‑security framework (RBI Directions 2026)**  

- **Overall IT‑Governance Framework**  
  - Must be **robust** and built around the five focus areas: strategic alignment, risk management, resource management, performance management, and Business Continuity/Disaster‑Recovery [Source 2].  
  - Must **specify the governance structure and processes** needed to meet the bank’s strategic objectives [Source 2].  
  - Must **define the roles, authority and responsibilities** of the Board of Directors (or Board‑level committee) and senior management [Source 2].  
  - Must include **adequate oversight mechanisms** to ensure accountability and mitigation of IT and cyber‑information‑security risks [Source 2].

- **Accountability and Effectiveness**  
  - The IT‑governance and information‑security governance structure must **foster accountability**, be **effective and efficient**, have **adequate skilled resources**, clear objectives and **unambiguous responsibilities at every organisational level** [Source 3].  
  - The bank must **establish processes for assessing and managing IT and cyber‑security risks** [Source 3].  
  - **Budgetary allocations** for IT and cyber‑security must be **commensurate with the bank’s IT maturity, digital depth, threat environment and industry standards**, and used to meet the stated objectives [Source 3].  
  - **Organisational arrangements** must ensure security concerns receive appropriate attention and are **escalated promptly** to enable quick action [Source 3].

- **Governance of the Cyber‑Security Operations Centre (CSOC)**  
  - The bank must set up a **CSOC framework** proportionate to its technology‑risk profile, scale, complexity, business needs and regulatory/legal obligations [Source 1].  
  - Governance for the CSOC must include:  
    - **Briefing the Board/ITSC and senior management** on threat intelligence [Source 1].  
    - **Dashboards** for effective oversight [Source 1].  
    - **Formulation and enforcement of policies** with defined metrics, reporting structures and clear reporting requirements [Source 1].  
    - **Timely communication** with all stakeholders [Source 1].

- **Policy Compliance and Review**  
  - All staff and service providers must **comply with information‑security and acceptable‑use policies** [Source 5].  
  - The bank must **review its security infrastructure and policies at least annually**, incorporating experience and emerging threats [Source 5].  

These elements together constitute the RBI‑mandated governance requirements that a bank must embed in its technology‑risk and cyber‑security framework.

### Sources

,source,document,type,year,page
0,Source 1,file5.pdf,other,2026,50
1,Source 2,file5.pdf,other,2026,12
2,Source 3,file5.pdf,other,2026,14
3,Source 4,file5.pdf,other,2026,22
4,Source 5,file5.pdf,other,2026,21


In [24]:
_ = ask("When must a bank classify a loan account as a non-performing asset?")

### Answer

**When a bank must classify a loan account as a Non‑Performing Asset (NPA)**  

*According to the RBI “Commercial Banks‑Asset Classification, Provisioning and Income Recognition” Directions (2026):*

- **General rule** – The account is classified as NPA **as soon as any one of the following conditions is met** and the bank’s day‑end processing reflects that status [Source 1].

- **Specific trigger conditions**  
  1. **Overdue interest and/or principal for more than 90 days** on a term loan, purchased bill or discounted bill.  
  2. **Overdraft / Cash Credit (OD/CC) marked “out of order.”**  
  3. **Continuous drawings for 90 days** on an OD/CC where the sanctioned drawing power is based on stock or receivable statements that are older than three months.  
  4. **Crop‑loan defaults** –  
     - For short‑duration crops: overdue for **two consecutive crop seasons** (e.g., rabi‑rabi‑rabi).  
     - For long‑duration crops: overdue for **one crop season**.  

- **Timing of classification** – The NPA status takes effect on the **day‑end processing date** when the condition is satisfied. For example, if a loan’s due date is 31 Mar 2021 and the full amount is not received before the day‑end run, the overdue date is recorded as 31 Mar 2021; the account is then classified as NPA on the next day‑end run after the 90‑day (or other) threshold is crossed (illustrated as 29 Jun 2021) [Source 2].

**In short:** A loan becomes an NPA the moment any of the above overdue or “out‑of‑order” conditions is met, and the classification is recorded on the bank’s day‑end processing date. [Source 1][Source 2]

### Sources

,source,document,type,year,page
0,Source 1,file4.pdf,other,2026,10
1,Source 2,file4.pdf,other,2026,17
2,Source 3,file4.pdf,other,2026,16
3,Source 4,file4.pdf,other,2026,5
4,Source 5,file4.pdf,other,2026,45


In [25]:
_ = ask("What are the functions and powers of the Internal Ombudsman of a bank?")

### Answer

**Functions and powers of the Internal Ombudsman (IO) of a bank (as per the 2025 RBI Directions)**  

- **Scope of complaints handled**  
  - May only consider complaints that have already been examined by the bank and are either *partially resolved* or *wholly rejected* by the bank; it does **not** receive complaints directly from the public. [Source 1]

- **Decision‑making authority**  
  - The IO can *decide* on any or all such complaints.  
  - The Deputy IO’s (Dy. IO) authority to decide complaints is defined by a policy approved by the bank’s Customer Service Committee. [Source 1]

- **Compensation recommendation**  
  - The IO/Dy. IO *recommends suitable compensation* to the complainant, following the compensation limits prescribed by RBI guidelines (if any) and the bank’s own compensation policy (if any). [Source 1]

- **Legal representation**  
  - The IO/Dy. IO **shall not** represent the bank in any legal proceedings before courts, tribunals, or other authorities. [Source 1]

- **Reporting lines**  
  - **Administrative reporting:** to the bank’s *Competent Authority* as defined under the Directions. [Source 2]  
  - **Functional reporting:** to the *Customer Service Committee* of the Board. [Source 2]

- **Support and infrastructure**  
  - The bank must provide the IO with sufficient officers, staff, IT support, and office space (preferably at the Head Office or Corporate Office) to enable effective discharge of duties. [Source 2]

- **Audit and oversight**  
  - The bank’s Internal Audit Department conducts an annual audit of the implementation of the Directions, covering the IO’s functioning. [Source 2]

These points capture the core functions (reviewing and deciding on bank‑examined complaints, recommending compensation) and powers (decision‑making authority, reporting, and independence from legal representation) of the Internal Ombudsman as outlined in the 2025 RBI regulatory framework.

### Sources

,source,document,type,year,page
0,Source 1,file3.pdf,other,2025,8
1,Source 2,file3.pdf,other,2025,6
2,Source 3,file3.pdf,other,2025,4
3,Source 4,file3.pdf,other,2025,5
4,Source 5,file3.pdf,other,2025,15


In [26]:
_ = ask("What are the requirements for processing recurring payments under the e-mandate framework?")

### Answer

**Requirements for processing recurring payments under the e‑mandate framework (RBI Directions 2026)**  

- **Scope of applicability** – The directions apply to *all* Payment System Providers (PSPs) and Payment System Participants for processing recurring transactions (domestic or cross‑border) using cards, PPI or UPI. [Source 2]  

- **Mandate set‑up and customer communication**  
  - The e‑mandate must specify a validity period and allow the customer to withdraw it at any time; this facility must be clearly communicated at registration. [Source 3]  

- **Amount specifications**  
  - The mandate may be for a *fixed* amount **or** a *variable* amount subject to the overall RBI‑defined cap.  
  - For variable‑amount mandates, the issuer must give the customer a way to set the *maximum value* of any recurring transaction. [Source 3]  

- **Pre‑transaction notification**  
  - The customer must be provided a facility to choose or change the mode (e.g., SMS, email, etc.) for receiving pre‑transaction notifications from the issuer. [Source 3]  

- **Authentication & validation**  
  - **First transaction** under an e‑mandate must undergo AFA (Authentication Factor Authorization) validation. If the first transaction is processed together with registration, the same AFA validation applies. [Source 3]  
  - Any **modification or withdrawal** of an existing e‑mandate also requires AFA validation by the issuer. [Source 3]  

- **Cost to the customer**  
  - No charges may be levied on the customer for using the e‑mandate facility for recurring transactions. [Source 4]  

- **Card‑specific provision**  
  - Existing e‑mandates can be mapped to re‑issued cards. [Source 4]  

- **Acquirer responsibility**  
  - Acquirers must ensure that merchants they onboard comply with these directions. [Source 4]  

### Sources

,source,document,type,year,page
0,Source 1,file2.pdf,other,2026,3
1,Source 2,file2.pdf,other,2026,1
2,Source 3,file2.pdf,other,2026,2
3,Source 4,file2.pdf,other,2026,4
4,Source 5,file2.pdf,other,2026,4


In [27]:
_ = ask("What conditions must an entity meet to be authorised to operate a payment system?")

### Answer

**Conditions an entity must satisfy to obtain RBI authorisation to operate a payment system (as per the 2026 RBI directions)**  

- **Submit a formal application** – The entity must file an application in the prescribed form through the RBI’s online portal. [Source 1]  
- **Meet the capital requirement** – The capital to be held must be in accordance with the specific Guidelines/Directions issued for the payment system concerned (details are published on the RBI website). [Source 1]  
- **Maintain the required net‑worth** – Net‑worth is defined as the sum of paid‑up equity capital, preference shares compulsorily convertible to equity, free reserves, share‑premium balance and capital reserves (excluding revaluation reserves), **adjusted** for accumulated losses, intangible assets, deferred revenue expenditure and deferred tax assets, if any. [Source 1]  
- **Be free from a cooling‑off restriction** – Entities that have had a Certificate of Authorisation (CoA) voluntarily surrendered, whose authorisation was refused, or that are linked to promoters who fall under the disqualifying categories listed in the circular, must observe a cooling‑off period during which they cannot submit a new application. [Source 3]  
- **Comply with promoter‑eligibility rules** – A promoter that is a body corporate or an individual must not be deemed a “promoter group” merely because a financial institution, scheduled commercial bank, foreign institutional investor or mutual fund holds ≥10 % of its equity, unless that holding is strategic. [Source 4]  

*Only entities that satisfy all the above conditions are eligible to be granted an Authorisation (Certificate of Authorisation) to operate a payment system under the Payment and Settlement Systems Act, 2026.*

### Sources

,source,document,type,year,page
0,Source 1,file1.pdf,other,2026,4
1,Source 2,file1.pdf,other,2026,7
2,Source 3,file1.pdf,other,2026,10
3,Source 4,file1.pdf,other,2026,4
4,Source 5,file1.pdf,other,2026,8


Cell 2: cross-document and reasoning questions

These test whether hybrid retrieval pulls from more than one document.

In [28]:
_ = ask("How do the fraud risk management and cybersecurity directions differ in their incident reporting expectations?")

### Answer

**Fraud‑Risk‑Management Directions (2022 RBI Circular)**  

- **What must be reported** – Every individual fraud case, regardless of amount, is to be reported to RBI through a **Fraud Monitoring Return (FMR)**. The bank must also report payment‑system‑related disputed or suspected fraudulent transactions to the **Central Payments Fraud Information Registry (CPFIR)** and, if later confirmed as fraud, again through the FMR [Source 2].  
- **Reporting timeline** – The initial FMR must be filed **immediately and no later than 14 days** from the date the incident/account is classified as fraud. Updates are submitted via the **FMR Update Application (FUA)** [Source 2].  
- **Reporting format** – The bank selects one of the prescribed fraud categories (e.g., misappropriation of funds, forged instruments, manipulation of books, etc.) when filing the FMR [Source 3].  
- **Other requirements** – No monthly fraud certificates or flash reports are required; the focus is on a single, timely electronic return for each case [Source 2].

**Cybersecurity Directions (2026 RBI Framework)**  

- **What must be reported** – The Authorized Service Provider (ASP) must have a **documented incident‑management procedure** and must **inform the bank of any cyber‑incident** affecting the bank, so that the bank can take mitigation steps and satisfy regulatory obligations [Source 5].  
- **Reporting timeline** – The wording is “on a **timely basis**,” without a fixed number of days; the emphasis is on prompt notification rather than a statutory deadline.  
- **Reporting format & scope** – Reporting is part of a broader incident‑management and Business Continuity/Disaster‑Recovery (BCP/DR) process, which includes maintaining audit logs, alert mechanisms for log‑setting changes, and coordination with third‑party vendors [Source 5].  

### Key Differences in Incident‑Reporting Expectations  

| Aspect | Fraud‑Risk‑Management Directions | Cybersecurity Directions |
|--------|--------------------------------|--------------------------|
| **Trigger** | Any confirmed fraud case (individual) | Any cyber‑incident affecting the bank |
| **Reporting entity** | Direct filing to RBI via FMR (and CPFIR for payment‑system disputes) | Notification to the bank (which then complies with RBI’s broader cyber‑risk framework) |
| **Timeline** | **≤ 14 days** from fraud classification (mandatory) | **Timely** (no explicit deadline) |
| **Format** | Structured FMR with predefined fraud categories | Incident‑management procedure; audit logs and alerts; no single prescribed form |
| **Scope of requirement** | Individual case‑by‑case reporting; no monthly certificates needed | Ongoing monitoring, log capture, and coordination with third‑party service providers; part of BCP/DR compliance |

Thus, fraud‑risk directions prescribe a **specific, deadline‑driven electronic return for each fraud incident**, whereas cybersecurity directions call for **prompt, procedural notification to the bank without a fixed deadline**, embedded within a broader incident‑management and resilience framework.

### Sources

,source,document,type,year,page
0,Source 1,file7.pdf,other,2022,3
1,Source 2,file7.pdf,other,2022,19
2,Source 3,file7.pdf,other,2022,18
3,Source 4,file7.pdf,other,2022,20
4,Source 5,file5.pdf,other,2026,38


In [29]:
_ = ask("Which directions require banks to have an independent audit or review function, and what do they require?")

### Answer

**Directions that mandate an independent audit / review function**

| Direction (year) | Independent audit / review function required | What the direction requires |
|------------------|----------------------------------------------|-----------------------------|
| **RBI (CBs – Cybersecurity, Technology: Risk, Resilience and Assurance Framework) Directions, 2026** – *Chapter VII – Information Systems Audit*【Source 1】 | **Separate Information‑Systems (IS) Audit function** (or dedicated resources within the Internal Audit function). | • The bank must establish an IS‑Audit policy covering mandate, purpose, authority, audit universe and periodicity, approved by the Audit Committee of the Board (ACB) and reviewed at least annually.<br>• The ACB oversees IS‑Audit, reviews critical IT/cyber‑security issues and gives direction to management.<br>• Where external resources are used for IS‑audit, they must possess the required professional skills and competence. |
| **RBI (Commercial Banks – Asset Classification, Provisioning and Income Recognition) Directions, 2026** – *Three‑tier model risk‑management framework*【Source 3】 | **Internal Audit** as the third tier. | • Internal Audit shall provide **objective assurance** on the model‑risk ecosystem (e.g., validation of ECL models, monitoring of model performance).<br>• This assurance is independent of the front‑line and risk‑management functions. |
| **RBI (CBs – Fraud Risk Management) Directions, 2026** (document dated 2022)【Source 5】 | **External audit or internal audit** for “red‑flagged” (fraud‑suspected) loan accounts. | • For any credit facility/loan classified as a red‑flagged account, the bank must commission an **external audit** or an **internal audit** (as per its Board‑approved policy) to investigate the account.<br>• The bank must also frame a **policy on engagement of external auditors**, covering due‑diligence, competency, track‑record, and contractual clauses on audit timelines and report submission, approved by the Board. |
| **RBI (CBs – Cybersecurity, Technology: Risk, Resilience and Assurance Framework) Directions, 2026** – *Vulnerability Assessment / Penetration Testing (VA/PT) audit*【Source 4】 | **Independent VA/PT auditor** (external). | • The bank must ensure that the VA/PT audit report explicitly provides reasonable assurance for each area covered.<br>• At the time of empanelling or awarding the VA/PT contract, the requirement for independent assurance must be stated.<br>• Ongoing performance of the VA/PT auditor must be reviewed, and corrective action taken if deficiencies are identified. |

**Summary**

- The **Cybersecurity (2026) Directions** require a dedicated IS‑Audit function (or suitably skilled external resources) with a Board‑approved policy and ACB oversight.  
- The **Asset Classification (2026) Directions** embed internal audit as an independent assurance tier for model risk.  
- The **Fraud Risk Management (2026) Directions** compel banks to obtain an independent audit—external or internal—when a loan is red‑flagged, and to maintain a Board‑approved policy governing external auditor engagement.  
- The **Cybersecurity (2026) Directions** also mandate the use of an independent VA/PT auditor and continuous performance monitoring of that auditor.

### Sources

,source,document,type,year,page
0,Source 1,file5.pdf,other,2026,54
1,Source 2,file5.pdf,other,2026,29
2,Source 3,file4.pdf,other,2026,33
3,Source 4,file5.pdf,other,2026,42
4,Source 5,file7.pdf,other,2022,12


In [30]:
_ = ask("What customer protection measures appear across the digital payment and ombudsman directions?")

### Answer

The provided documents do not contain enough information to answer this question.

### Sources

,source,document,type,year,page
0,Source 1,file6.pdf,other,2026,16
1,Source 2,file6.pdf,other,2026,6
2,Source 3,file6.pdf,other,2026,10
3,Source 4,file6.pdf,other,2026,9
4,Source 5,file6.pdf,other,2026,15


Cell 3: metadata filter tests

In [31]:
# Should only return 2026 RBI directions
_ = ask("What are the requirements for fraud monitoring?", doc_type="circular", min_year=2026)

### Answer

The provided documents do not contain enough information to answer this question.

Cell 4: out-of-scope questions (the bot should refuse)

In [32]:
_ = ask("Who won the 2011 Cricket World Cup?")

### Answer

The provided documents do not contain enough information to answer this question.

### Sources

,source,document,type,year,page
0,Source 1,file5.pdf,other,2026,1
1,Source 2,file1.pdf,other,2026,7
2,Source 3,file5.pdf,other,2026,10
3,Source 4,file5.pdf,other,2026,54
4,Source 5,file1.pdf,other,2026,3


In [33]:
# Finance-related but not in these documents, a harder refusal test
_ = ask("What is the current repo rate set by the RBI?")

### Answer

The provided documents do not contain enough information to answer this question.

### Sources

,source,document,type,year,page
0,Source 1,file4.pdf,other,2026,62
1,Source 2,file1.pdf,other,2026,10
2,Source 3,file6.pdf,other,2026,29
3,Source 4,file4.pdf,other,2026,78
4,Source 5,file1.pdf,other,2026,1


## 11. Evaluation
Without numbers, a RAG project is just a demo. This section builds a **synthetic test set** and measures:

| Metric | What it tells you |
|---|---|
| **Hit rate @k** | Did the chunk the question was written from appear in the top-k results? |
| **MRR** | How high up was it? (1.0 = always first) |
| **Faithfulness** | Does an LLM judge find every claim in the answer supported by the retrieved context? |
| **Refusal rate (out-of-scope)** | Does the bot correctly decline questions the documents can't answer? |

Test questions are generated by the LLM from randomly sampled chunks, so you don't have to write them by hand. Skim them for quality. For a stronger project, hand-write 10 of your own and mix them in.

In [34]:
def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

qgen_prompt = ChatPromptTemplate.from_template("""
You are building a test set for a banking/finance document assistant.
Read the passage and write ONE specific question that can be answered using ONLY this passage.
The question must be self-contained: do not say "the passage", "this document" or "the text".
Prefer questions about concrete rules, limits, definitions, dates, requirements or figures.
Return JSON only, no extra text: {{"question": "...", "answer": "..."}}

Passage:
{passage}
""")
qgen_chain = qgen_prompt | llm | StrOutputParser()

N_EVAL = 20   # raise for a more reliable estimate (watch Groq rate limits)
random.seed(42)
candidates = [c for c in chunks if len(c.page_content) > 500]
sample = random.sample(candidates, min(N_EVAL, len(candidates)))

eval_set = []
for c in sample:
    out = extract_json(safe_invoke(qgen_chain, {"passage": c.page_content}))
    if out and out.get("question"):
        eval_set.append({"question": out["question"], "reference": out.get("answer", ""), "chunk_id": c.metadata["chunk_id"]})
    time.sleep(0.5)

print(f"Generated {len(eval_set)} test questions")
display(pd.DataFrame(eval_set)[["question", "reference"]].head(10))

Generated 20 test questions


,question,reference
0,What type of training must be provided to pers...,Specialised training.
1,By which date must the bank submit its quarter...,On or before the 15th day of the month followi...
2,How long is the renewal period granted to an e...,One year
3,What security measures must the bank implement...,The bank shall implement multi factor authenti...
4,What may a bank use when historical data on OD...,"The bank may use suitable benchmarks, which sh..."
5,Under what condition may the renewal date be c...,"When the bank is able to demonstrate, through ..."
6,When a bank uses its internal credit rating sy...,The internal policy must clearly define the nu...
7,What are the three key actions outlined in the...,"1) Expected Credit Loss (ECL) approach, while ..."
8,"According to the passage, what requirement mus...",At least one of the authentication methodologi...
9,What type of submission may the IO/Dy. IO requ...,"Written or oral submission, including addition..."


### 11a. Compare retrieval strategies

In [35]:
MODES = ["vector", "bm25", "hybrid", "hybrid+rerank"]

def retrieval_metrics(mode, k=TOP_K):
    hits, rr = 0, []
    for item in eval_set:
        results = retrieve(item["question"], mode=mode, k=k)
        ids = [d.metadata["chunk_id"] for d in results]
        if item["chunk_id"] in ids:
            hits += 1
            rr.append(1.0 / (ids.index(item["chunk_id"]) + 1))
        else:
            rr.append(0.0)
    n = len(eval_set)
    return {"mode": mode, f"hit_rate@{k}": round(hits / n, 3), "MRR": round(sum(rr) / n, 3)}

retrieval_results = pd.DataFrame([retrieval_metrics(m) for m in MODES])
display(retrieval_results)
best_mode = retrieval_results.sort_values("MRR", ascending=False).iloc[0]["mode"]
print("Best mode by MRR:", best_mode)

,mode,hit_rate@5,MRR
0,vector,0.90,0.617
1,bm25,0.95,0.883
2,hybrid,0.90,0.787
3,hybrid+rerank,0.95,0.892


Best mode by MRR: hybrid+rerank


### 11b. Faithfulness and refusal behaviour
An LLM judge checks whether each generated answer is fully supported by its retrieved context (a proxy for hallucination rate). Then we ask out-of-scope questions and measure how often the bot correctly refuses.

In [38]:
judge_prompt = ChatPromptTemplate.from_template("""
You are a strict fact-checker. Given CONTEXT and ANSWER, decide whether every factual claim in ANSWER
is directly supported by CONTEXT. Ignore citation formatting.
Return JSON only: {{"faithful": true or false, "reason": "one short sentence"}}

CONTEXT:
{context}

ANSWER:
{answer}
""")
judge_chain = judge_prompt | llm | StrOutputParser()

N_JUDGE = min(10, len(eval_set))
faith_rows = []
for item in eval_set[:N_JUDGE]:
    out = answer_question(item["question"], mode=best_mode)
    verdict = extract_json(safe_invoke(judge_chain, {"context": format_docs(out["docs"]), "answer": out["answer"]})) or {}
    faith_rows.append({
        "question": item["question"],
        "answered": REFUSAL not in out["answer"],
        "faithful": bool(verdict.get("faithful")),
        "reason": verdict.get("reason", ""),
    })
    time.sleep(0.5)

faith_df = pd.DataFrame(faith_rows)
display(faith_df)

# Out-of-scope questions: the correct behaviour is to refuse
OUT_OF_SCOPE = [
    "Who won the 2011 Cricket World Cup?",
    "What is the recipe for butter chicken?",
    "How do I train a neural network in PyTorch?",
    "What is the capital of Australia?",
    "Who is the CEO of Tesla?",
    "What will the Nifty 50 close at next Friday?",
    "Write me a poem about the monsoon.",
    "What is the best smartphone to buy this year?",
]
oos_rows = []
for q in OUT_OF_SCOPE:
    out = answer_question(q, mode=best_mode)
    oos_rows.append({"question": q, "refused": REFUSAL in out["answer"]})
    time.sleep(0.5)
oos_df = pd.DataFrame(oos_rows)
display(oos_df)

,question,answered,faithful,reason
0,What type of training must be provided to pers...,True,True,The claim matches the statement in Source 1 ex...
1,By which date must the bank submit its quarter...,True,True,The statement matches the reporting deadline d...
2,How long is the renewal period granted to an e...,True,True,Both statements are directly supported by Sour...
3,What security measures must the bank implement...,True,True,All claims are directly backed by Source 1's s...
4,What may a bank use when historical data on OD...,True,True,The answer correctly restates the Source 1 tex...
5,Under what condition may the renewal date be c...,True,True,All claims are directly quoted or paraphrased ...
6,When a bank uses its internal credit rating sy...,True,True,All claims are directly stated in Source 1 of ...
7,What are the three key actions outlined in the...,True,False,"The answer adds definitions of PD, LGD, and EA..."
8,"According to the passage, what requirement mus...",True,True,All statements are directly quoted from Source 1.
9,What type of submission may the IO/Dy. IO requ...,True,True,The context explicitly states the IO/Dy. IO ma...


,question,refused
0,Who won the 2011 Cricket World Cup?,True
1,What is the recipe for butter chicken?,True
2,How do I train a neural network in PyTorch?,True
3,What is the capital of Australia?,True
4,Who is the CEO of Tesla?,True
5,What will the Nifty 50 close at next Friday?,True
6,Write me a poem about the monsoon.,True
7,What is the best smartphone to buy this year?,True


### 11c. Summary: numbers for your resume

In [39]:
best_row = retrieval_results[retrieval_results["mode"] == best_mode].iloc[0]
baseline_row = retrieval_results[retrieval_results["mode"] == "vector"].iloc[0]
hit_col = f"hit_rate@{TOP_K}"

summary = {
    "documents": len({c.metadata["doc_name"] for c in chunks}),
    "chunks": len(chunks),
    "test_questions": len(eval_set),
    "baseline (vector) hit rate": baseline_row[hit_col],
    f"best ({best_mode}) hit rate": best_row[hit_col],
    "best MRR": best_row["MRR"],
    "faithfulness": round(faith_df["faithful"].mean(), 2),
    "out-of-scope refusal rate": round(oos_df["refused"].mean(), 2),
}
display(pd.Series(summary, name="value").to_frame())
print(f"Example resume line: 'Improved retrieval hit rate@{TOP_K} from {baseline_row[hit_col]:.0%} to "
      f"{best_row[hit_col]:.0%} with hybrid BM25 + vector search and cross-encoder reranking; "
      f"{summary['out-of-scope refusal rate']:.0%} of out-of-scope questions correctly refused.'")

,value
documents,7.000
chunks,570.000
test_questions,20.000
baseline (vector) hit rate,0.900
best (hybrid+rerank) hit rate,0.950
best MRR,0.892
faithfulness,0.900
out-of-scope refusal rate,1.000


Example resume line: 'Improved retrieval hit rate@5 from 90% to 95% with hybrid BM25 + vector search and cross-encoder reranking; 100% of out-of-scope questions correctly refused.'


## 12. Save and reload the index
So you don't re-embed every session.

In [40]:
vectorstore.save_local("findoc_faiss_index")
with open("findoc_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print("Saved to ./findoc_faiss_index and ./findoc_chunks.pkl")

# To reload in a new session (after re-running sections 2-3 and 7's embeddings/reranker setup):
# vectorstore = FAISS.load_local("findoc_faiss_index", embeddings, allow_dangerous_deserialization=True)
# chunks = pickle.load(open("findoc_chunks.pkl", "rb"))
# bm25 = BM25Retriever.from_documents(chunks, k=FETCH_K)

# Optional: persist to Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r findoc_faiss_index findoc_chunks.pkl /content/drive/MyDrive/

Saved to ./findoc_faiss_index and ./findoc_chunks.pkl


## 13. Gradio chat UI
Launches a shareable demo link you can put on your resume or README (the link lives while the notebook runs).

In [42]:
# import gradio as gr

# def gradio_ask(question, mode, doc_type, min_year):
#     out = answer_question(
#         question,
#         mode=mode,
#         doc_type=None if doc_type == "any" else doc_type,
#         min_year=int(min_year) if min_year else None,
#     )
#     src = sources_table(out["docs"]).to_markdown(index=False) if out["docs"] else "_No sources retrieved._"
#     return out["answer"], src

# demo = gr.Interface(
#     fn=gradio_ask,
#     inputs=[
#         gr.Textbox(label="Question", lines=2, placeholder="e.g. What are the KYC requirements for opening an account?"),
#         gr.Dropdown(MODES, value="hybrid+rerank", label="Retrieval mode"),
#         gr.Dropdown(["any", "circular", "annual_report", "faq", "fee_schedule", "other"], value="any", label="Document type"),
#         gr.Number(label="Minimum year (0 = no filter)", value=0, precision=0),
#     ],
#     outputs=[gr.Markdown(label="Answer"), gr.Markdown(label="Sources")],
#     title="FinDoc RAG: Banking & Regulatory Assistant",
#     description="Answers only from the loaded documents, with citations. Refuses out-of-scope questions.",
# )
# demo.launch(share=True)

## 14. How to describe this on your resume
> **FinDoc RAG Assistant** | LangChain, FAISS, BM25, sentence-transformers, Groq (gpt-oss-120b), Gradio
> - Built a retrieval-augmented assistant over RBI circulars and bank annual reports with page-level citations, metadata filtering (document type, year) and an explicit refusal policy
> - Implemented hybrid retrieval (vector + BM25 with Reciprocal Rank Fusion) and cross-encoder reranking; improved hit rate@5 from **X%** to **Y%** on a synthetic test set
> - Measured hallucination with an LLM-as-judge faithfulness check (**Z%** faithful) and out-of-scope refusal rate (**W%**)

Fill in X, Y, Z, W from the summary table in section 11c, and only use numbers you measured.

**Ideas to extend it:**
- Hand-write 10-15 gold questions and add them to the eval set
- Compare chunk sizes (500 / 1000 / 1500) and embedding models using the same eval
- Add conversation memory so follow-up questions work
- Move the pipeline into a FastAPI service and deploy it

FinDoc RAG Assistant | LangChain, FAISS, BM25, Sentence-Transformers, Groq (gpt-oss-120b)

Built a retrieval-augmented assistant over RBI circulars and bank annual reports with page-level citations, document metadata filtering, and an explicit out-of-scope refusal policy.

Implemented hybrid retrieval using vector search + BM25 with Reciprocal Rank Fusion, followed by cross-encoder reranking; improved Hit Rate@5 from 90% to 95% across a 20-question evaluation set.

Evaluated response reliability using an LLM-as-judge faithfulness check, achieving 90% faithfulness and correctly refusing 100% of out-of-scope queries.